# Homework 2 — Linear Regression

2026 cohort.  
Dataset: 2026 car fuel

# Setup and getting data

In [71]:
import pandas as pd
import numpy as np

df = pd.read_csv("../cohorts/2026/data/car_fuel_efficiency_2026.csv")
df.head()

,model_year,origin,fuel_type,drivetrain,num_doors,engine_displacement,num_cylinders,horsepower,vehicle_weight,acceleration,fuel_efficiency_mpg
0,2006,Europe,Gasoline,Front-wheel drive,4,2180,6,243.0,3870,NaN,31.9
1,2008,Europe,Diesel,Front-wheel drive,4,2390,6,272.0,4210,NaN,31.3
2,1996,Asia,Gasoline,Front-wheel drive,5,2320,6,267.0,4240,17.3,27.5
3,1989,Europe,Gasoline,Front-wheel drive,4,2130,6,258.0,4490,18.7,28.5
4,1994,USA,Diesel,Front-wheel drive,3,2580,7,304.0,4510,17.5,31.0


# Question 1 Finding column

There's one column with missing values. What is it?

In [72]:
df.isnull().sum()

model_year               0
origin                   0
fuel_type                0
drivetrain               0
num_doors                0
engine_displacement      0
num_cylinders            0
horsepower             877
vehicle_weight           0
acceleration           264
fuel_efficiency_mpg      0
dtype: int64

horsepower


# Question 2 Median

What's the median (50% percentile) for variable 'horsepower'?

In [73]:
df['horsepower'].median()

np.float64(254.0)

254

# Prepare and split the dataset

Shuffle the filtered dataset and create the split exactly as in the lecture:

In [74]:
n = len(df)
n_val = int(n * 0.2)
n_test = int(n * 0.2)
n_train = n - n_val - n_test

np.random.seed(42)
idx = np.arange(n)
np.random.shuffle(idx)

df_train = df.iloc[idx[:n_train]]
df_val = df.iloc[idx[n_train:n_train + n_val]]
df_test = df.iloc[idx[n_train + n_val:]]

# Question 3 Comparing filling

Train linear regression **without regularization**.

Fill missing `horsepower` with 0 first, then with the training mean, and compare validation RMSE.

In [75]:
base = [
    'engine_displacement',
    'horsepower',
    'vehicle_weight',
    'model_year',
]

y_train = df_train['fuel_efficiency_mpg'].values
y_val = df_val['fuel_efficiency_mpg'].values

X_train = df_train[base].fillna(0).values
X_val = df_val[base].fillna(0).values

In [76]:
mean = df_train[base].mean()
X_train_mean = df_train[base].fillna(mean).values
X_val_mean = df_val[base].fillna(mean).values


In [77]:
def train_linear_regression(X, y):
    ones = np.ones(X.shape[0])
    X = np.column_stack([ones, X])

    XTX = X.T.dot(X)
    XTX_inv = np.linalg.inv(XTX)
    w_full = XTX_inv.dot(X.T).dot(y)

    return w_full[0], w_full[1:]


def rmse(y, y_pred):
    se = (y - y_pred) ** 2
    mse = se.mean()
    return np.sqrt(mse)

## Without regularization

Fill missing values with 0. No regularization (`r = 0`).

In [78]:
w0, w = train_linear_regression(X_train, y_train)
y_pred = w0 + X_val.dot(w)

score = rmse(y_val, y_pred)
print(score)

2.2052931947548506


Fill missing values with mean. No regularization (`r = 0`).

In [79]:
w0, w = train_linear_regression(X_train_mean, y_train)
y_pred = w0 + X_val_mean.dot(w)

score = rmse(y_val, y_pred)
print(score)

2.2018174845658636


Fill with mean perform better in RMSE

# Question 4 Regularzation

Train linear regression **with regularization**.



fill the NAs with 0.

In [80]:
X_train = df_train[base].fillna(0).values
X_val = df_val[base].fillna(0).values

In [81]:
def train_linear_regression_reg(X, y, r=0.001):
    ones = np.ones(X.shape[0])
    X = np.column_stack([ones, X])

    XTX = X.T.dot(X)
    XTX = XTX + r * np.eye(XTX.shape[0])

    XTX_inv = np.linalg.inv(XTX)
    w_full = XTX_inv.dot(X.T).dot(y)

    return w_full[0], w_full[1:]

Try different values of r from this list: [0, 0.01, 0.1, 1, 5, 10, 100]

In [82]:
for r in [0, 0.01, 0.1, 1, 5, 10, 100]:
    w0, w = train_linear_regression_reg(X_train, y_train, r=r)
    y_pred = w0 + X_val.dot(w)
    score = round(rmse(y_val, y_pred), 4)
    print(r, score)

0 2.2053
0.01 2.2058
0.1 2.2241
1 2.3492
5 2.4094
10 2.4195
100 2.4292


The best validation RMSE is r = 0.

# Question 5 Trying different seed values

In [83]:
n = len(df)
n_val = int(n * 0.2)
n_test = int(n * 0.2)
n_train = n - n_val - n_test

In [ ]:
scores = []
for seed in [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]:
    np.random.seed(seed)
    idx = np.arange(n)
    np.random.shuffle(idx)

    df_train = df.iloc[idx[:n_train]]
    df_val = df.iloc[idx[n_train:n_train + n_val]]

    X_train = df_train[base].fillna(0).values
    X_val = df_val[base].fillna(0).values

    y_train = df_train['fuel_efficiency_mpg'].values
    y_val = df_val['fuel_efficiency_mpg'].values
    
    w0, w = train_linear_regression(X_train, y_train)
    y_pred = w0 + X_val.dot(w)
    scores.append(rmse(y_val, y_pred))

In [89]:
std = np.std(scores)
print(scores)
print(round(std, 3))

[np.float64(2.239204143051171), np.float64(2.2021128210833383), np.float64(2.163419778753089), np.float64(2.2043588768501907), np.float64(2.201880094330856), np.float64(2.2457851509085045), np.float64(2.2697212079514624), np.float64(2.19079459993641), np.float64(2.216611201697255), np.float64(2.2070365928266966)]
0.029


0.029

# Question 6

Split with seed 9. Combine train and validation, fill missing values with 0, and train with `r=0.001`. What's the RMSE on the test dataset?

In [100]:
scores = []
for seed in [9]:
    np.random.seed(seed)
    idx = np.arange(n)
    np.random.shuffle(idx)

    df_train = df.iloc[idx[:n_train + n_val]]
    df_test = df.iloc[idx[n_train + n_val:]]

    X_train = df_train[base].fillna(0).values
    y_train = df_train['fuel_efficiency_mpg'].values

    X_test = df_test[base].fillna(0).values
    y_test = df_test['fuel_efficiency_mpg'].values

    w0, w = train_linear_regression_reg(X_train, y_train, r=0.001)
    y_pred = w0 + X_test.dot(w)
    scores.append(rmse(y_test, y_pred))

print(scores)

[np.float64(2.2358277471945915)]


2.236